## Disease Symptoms Dataset (EDA)
- Goal: understand a symptoms -> disease dataset.
- Simple checks + a few plots + (optional) a tiny baseline model.


In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

train_path = Path('datasets/Training.csv')
test_path = Path('datasets/Testing.csv')

uploaded = None
if not train_path.exists():
    from google.colab import files
    uploaded = files.upload()

    def _find_key(part: str):
        part = part.lower()
        for k in uploaded.keys():
            if part in k.lower():
                return k
        return None

    train_key = _find_key('training') or list(uploaded.keys())[0]
    test_key = _find_key('testing')

    train_path = Path(train_key)
    test_path = Path(test_key) if test_key else None

print('train_path:', train_path)
print('test_path:', test_path)

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path) if test_path else None

train_df.head()


### Clean columns
- The training file sometimes has an extra empty column because of a trailing comma.


In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    drop_cols = [c for c in df.columns if c == '' or c.lower().startswith('unnamed')]
    if drop_cols:
        df = df.drop(columns=drop_cols)
    return df

train_df = clean_columns(train_df)
if test_df is not None:
    test_df = clean_columns(test_df)

print('Train shape:', train_df.shape)
if test_df is not None:
    print('Test shape:', test_df.shape)

train_df.columns[-5:]


### First look
- Quick look at types and a sample.


In [ ]:
train_df.info()
train_df.head()


### Missing values
- Check if anything is missing.


In [ ]:
train_df.isna().sum().sort_values(ascending=False).head(10)


### Label balance
- How many diseases and how balanced are they?


In [ ]:
target = 'prognosis'
print('Unique diseases:', train_df[target].nunique())

vc = train_df[target].value_counts()
display(vc.head(10))

plt.figure(figsize=(10, 4))
vc.head(10).plot(kind='bar')
plt.title('Top 10 diseases (train)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


### Symptoms per patient
- Each row is a set of symptoms (0/1). I counted how many symptoms are marked as 1.


In [ ]:
symptom_cols = [c for c in train_df.columns if c != target]
train_df['symptom_count'] = train_df[symptom_cols].sum(axis=1)

display(train_df['symptom_count'].describe())

plt.figure(figsize=(8, 4))
sns.histplot(train_df['symptom_count'], bins=15)
plt.title('Symptoms per patient')
plt.tight_layout()
plt.show()


### Common symptoms
- Which symptoms appear most often in the training set?


In [ ]:
top_symptoms = train_df[symptom_cols].sum().sort_values(ascending=False).head(10)
display(top_symptoms)

plt.figure(figsize=(10, 4))
sns.barplot(x=top_symptoms.values, y=top_symptoms.index, color='steelblue')
plt.title('Top 10 symptoms (count of 1s)')
plt.xlabel('Count')
plt.ylabel('')
plt.tight_layout()
plt.show()


### Optional: tiny baseline model
- Not the final model, just a quick baseline to see if the data is learnable.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score

X = train_df[symptom_cols]
y = train_df[target].astype(str)

le = LabelEncoder()
y_enc = le.fit_transform(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)

model = BernoulliNB()
model.fit(X_train, y_train)

pred = model.predict(X_val)
print('Validation accuracy:', round(accuracy_score(y_val, pred), 4))

sample = X_val.iloc[[0]]
proba = model.predict_proba(sample)[0]
top3 = np.argsort(proba)[::-1][:3]
print('Top-3 predictions for 1 sample:')
for i in top3:
    print('-', le.inverse_transform([i])[0], f'{proba[i]*100:.1f}%')

if test_df is not None and target in test_df.columns:
    X_test = test_df.reindex(columns=symptom_cols, fill_value=0)
    y_test = le.transform(test_df[target].astype(str))
    print('Test accuracy (small test set):', round(accuracy_score(y_test, model.predict(X_test)), 4))


### Short findings
- Train set: 4,920 rows, 132 symptom features, 41 diseases (balanced: 120 each).
- Symptoms per patient: mean 7.45, median 6, usually between 5 and 10.
- Most common symptoms in this file: fatigue, vomiting, high_fever, loss_of_appetite, nausea.
- I dropped one empty column in `Training.csv` caused by a trailing comma.
